# Notebook 49 — Meta-Law Generalization with Holdout-Regime Tests

This notebook evaluates whether the metadata-conditioned meta-law generalizes to **unseen full regimes**.

For each held-out regime, it:
- trains the meta-law on all remaining regimes
- predicts the held-out coefficient vector from metadata
- reconstructs the governing field
- reconstructs trajectory families
- compares against a shared-law baseline and the regime-specific fit

## Fixed governing template

\[
g(r,c;\beta) = \beta_0 + \beta_1 c + \beta_2 r + \beta_3 c^3 + \beta_4 r^2 + \beta_5 r c^2
\]

## Main questions

1. Does the meta-law beat a shared-law on held-out regimes?
2. Which meta-model wins most often on true holdout?
3. How much generalization is retained at the field and trajectory level?

In [ ]:
# Core imports

import warnings
warnings.filterwarnings("ignore")

import os
import glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import r2_score, mean_squared_error
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neural_network import MLPRegressor

try:
    from IPython.display import display
except Exception:
    display = print

np.random.seed(42)

In [ ]:
# Data discovery and synthetic fallback

DATA_PATH = None

def autodetect_data_path():
    candidates = []
    patterns = [
        "*residual*flow*.parquet",
        "*residual*flow*.csv",
        "*governing*flow*.parquet",
        "*governing*flow*.csv",
        "*.parquet",
        "*.csv",
        "*.pkl",
        "*.pickle",
    ]
    for pat in patterns:
        candidates.extend(glob.glob(pat))
        candidates.extend(glob.glob(os.path.join("data", pat)))
        candidates.extend(glob.glob(os.path.join("outputs", pat)))
    candidates = [c for c in candidates if os.path.isfile(c)]
    return candidates[0] if candidates else None

def synthetic_dataset():
    systems = ["entropy", "unevenness"]
    tasks = ["zeta_vs_gue", "zeta_vs_poisson"]
    forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
    ks = [3, 5, 7]
    flow_modes = ["linear", "nonlinear"]

    rows = []
    sample_id = 0
    for system in systems:
        for task in tasks:
            for forcing_mode in forcing_modes:
                for k in ks:
                    for flow_mode in flow_modes:
                        n_paths = 14
                        n_steps = 42
                        c_grid = np.linspace(-1.25, 1.05, n_steps)

                        sys_shift = 0.06 if system == "entropy" else -0.04
                        task_shift = 0.05 if task == "zeta_vs_gue" else -0.03
                        force_shift = {"capacity_gap": 0.00, "feature_gap": 0.03, "condition_gap": 0.08}[forcing_mode]
                        k_shift = {3: -0.04, 5: 0.02, 7: 0.05}[k]
                        flow_shift = 0.05 if flow_mode == "nonlinear" else -0.02
                        nl_gain = 1.0 if flow_mode == "nonlinear" else 0.55

                        for path_id in range(n_paths):
                            r = -0.75 + 0.10 * path_id + 0.05 * np.sin(0.7 * path_id)
                            for window_id, c in enumerate(c_grid):
                                g = (
                                    0.58 * np.tanh(1.35 * c)
                                    + 0.42 * c
                                    - 0.78 * r
                                    + 0.20 * r**2
                                    + nl_gain * 0.07 * c**2
                                    + nl_gain * 0.10 * r * c
                                    - nl_gain * 0.025 * r**3
                                    + sys_shift + task_shift + force_shift + k_shift + flow_shift
                                )
                                if forcing_mode == "condition_gap":
                                    g += 0.06 * np.sin(2.3 * c)
                                if system == "entropy":
                                    g += 0.03 * np.cos(1.2 * c)
                                if task == "zeta_vs_poisson":
                                    g -= 0.015 * c
                                if flow_mode == "linear":
                                    g -= 0.09 * r**2
                                    g += 0.015 * c * r

                                delta_c = c_grid[min(window_id + 1, n_steps - 1)] - c if window_id < n_steps - 1 else c - c_grid[max(window_id - 1, 0)]
                                noise = 0.012 * np.random.randn()
                                next_r = r + (g + noise) * delta_c

                                rows.append(
                                    {
                                        "system": system,
                                        "task": task,
                                        "forcing_mode": forcing_mode,
                                        "k": k,
                                        "flow_mode": flow_mode,
                                        "condition_coord": c,
                                        "residual": r,
                                        "predicted_flow": g + noise,
                                        "next_residual": next_r,
                                        "delta_condition": delta_c,
                                        "sample_id": sample_id,
                                        "path_id": path_id,
                                        "window_id": window_id,
                                    }
                                )
                                r = next_r
                                sample_id += 1
    return pd.DataFrame(rows)

if DATA_PATH is None:
    DATA_PATH = autodetect_data_path()

USE_SYNTHETIC_FALLBACK = DATA_PATH is None
print("Selected DATA_PATH:", DATA_PATH)
print("USE_SYNTHETIC_FALLBACK:", USE_SYNTHETIC_FALLBACK)

In [ ]:
# Loading + schema alignment

def load_dataframe(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    return pd.read_csv(path)

alias_groups = {
    "condition_coord": ["condition_coord", "c", "condition", "cond", "condition_coordinate"],
    "residual": ["residual", "r", "resid"],
    "predicted_flow": ["predicted_flow", "flow", "g", "drdc", "delta_residual_per_condition"],
    "next_residual": ["next_residual", "r_next", "next_r"],
    "delta_condition": ["delta_condition", "dc", "delta_c"],
    "forcing_mode": ["forcing_mode", "forcing", "forcing_gap", "mode"],
}

def align_schema(df):
    cols = list(df.columns)
    rename_map = {}
    for canonical, aliases in alias_groups.items():
        for a in aliases:
            if a in cols and a != canonical:
                rename_map[a] = canonical
                break
    df = df.rename(columns=rename_map)

    if "next_residual" not in df.columns and {"residual", "predicted_flow", "delta_condition"}.issubset(df.columns):
        df["next_residual"] = df["residual"] + df["predicted_flow"] * df["delta_condition"]

    if "delta_condition" not in df.columns and "condition_coord" in df.columns:
        df = df.sort_values(["condition_coord"]).copy()
        dc = np.gradient(df["condition_coord"].to_numpy())
        df["delta_condition"] = dc

    defaults = {
        "system": "unknown_system",
        "task": "unknown_task",
        "forcing_mode": "unknown_forcing",
        "k": 5,
        "flow_mode": "unknown_flow",
        "sample_id": np.arange(len(df)),
        "path_id": 0,
        "window_id": np.arange(len(df)),
    }
    for k, v in defaults.items():
        if k not in df.columns:
            df[k] = v

    required = ["condition_coord", "residual", "predicted_flow"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after alignment: {missing}")
    return df

if USE_SYNTHETIC_FALLBACK:
    df = synthetic_dataset()
    data_source = "synthetic_fallback"
else:
    df = align_schema(load_dataframe(DATA_PATH))
    data_source = DATA_PATH

df = align_schema(df)
print("Data source:", data_source)
print("Shape:", df.shape)
display(df.head())

In [ ]:
# Fixed template and per-regime coefficient dataset

TERM_NAMES = ["1", "c", "r", "c^3", "r^2", "r c^2"]
COEF_COLS = [f"coef_{t}" for t in TERM_NAMES]

def design_template(data):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    return np.column_stack([
        np.ones_like(r),
        c,
        r,
        c**3,
        r**2,
        r * c**2,
    ])

def fit_template(sub, alpha=1e-6):
    X = design_template(sub)
    y = sub["predicted_flow"].to_numpy(dtype=float)
    beta = np.linalg.solve(X.T @ X + alpha * np.eye(X.shape[1]), X.T @ y)
    pred = X @ beta
    stats = {"n": len(sub), "r2": r2_score(y, pred), "rmse": math.sqrt(mean_squared_error(y, pred))}
    for name, coef in zip(TERM_NAMES, beta):
        stats[f"coef_{name}"] = float(coef)
    return beta, pred, stats

def predict_with_beta(sub, beta):
    return design_template(sub) @ beta

def safe_corr(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

def explained_variance_score_manual(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.var(y_true)
    if denom < 1e-12:
        return np.nan
    return float(1.0 - np.var(y_true - y_pred) / denom)

regime_rows = []
regime_subsets = {}

for vals, sub in df.groupby(["system", "task", "forcing_mode", "k", "flow_mode"]):
    if len(sub) < 30:
        continue
    beta, pred, stats = fit_template(sub.copy())
    kval = int(vals[3]) if float(vals[3]).is_integer() else vals[3]
    regime_id = f"{vals[0]}|{vals[1]}|{vals[2]}|k={kval}|{vals[4]}"
    row = {
        "regime_id": regime_id,
        "system": vals[0],
        "task": vals[1],
        "forcing_mode": vals[2],
        "k": float(vals[3]),
        "flow_mode": vals[4],
    }
    row.update(stats)
    regime_rows.append(row)
    regime_subsets[regime_id] = sub.copy()

coef_df = pd.DataFrame(regime_rows).reset_index(drop=True)
display(coef_df.head())
print("Number of regimes:", len(coef_df))

In [ ]:
# Metadata matrix

meta_df = coef_df[["regime_id", "system", "task", "forcing_mode", "flow_mode", "k"]].copy()
meta_X = pd.get_dummies(meta_df[["system", "task", "forcing_mode", "flow_mode"]], drop_first=False)
meta_X["k"] = coef_df["k"].astype(float).values
meta_X["k2"] = coef_df["k"].astype(float).values**2

X_meta = meta_X.copy()
Y_coef = coef_df[COEF_COLS].copy()

display(X_meta.head())
display(Y_coef.head())

In [ ]:
# Meta-model builders and helpers

class MultiTargetMLP:
    def __init__(self, hidden_layer_sizes=(16,), random_state=42, max_iter=4000):
        self.hidden_layer_sizes = hidden_layer_sizes
        self.random_state = random_state
        self.max_iter = max_iter
        self.models = None

    def fit(self, X, Y):
        Y = np.asarray(Y, dtype=float)
        if Y.ndim == 1:
            Y = Y[:, None]
        self.models = [
            MLPRegressor(hidden_layer_sizes=self.hidden_layer_sizes, random_state=self.random_state, max_iter=self.max_iter)
            for _ in range(Y.shape[1])
        ]
        for j, m in enumerate(self.models):
            m.fit(X, Y[:, j])
        return self

    def predict(self, X):
        cols = [m.predict(X) for m in self.models]
        return np.column_stack(cols)

def build_linear():
    return LinearRegression()

def build_ridge():
    return Ridge(alpha=1.0)

def build_mlp():
    return MultiTargetMLP()

def fit_predict_holdout(X_train, Y_train, X_test, model_builder):
    model = model_builder()
    model.fit(np.asarray(X_train, dtype=float), np.asarray(Y_train, dtype=float))
    return model.predict(np.asarray(X_test, dtype=float))

def trajectory_gap(sub, beta_ref, beta_cmp, n_r0=15):
    cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
    rmin, rmax = sub["residual"].min(), sub["residual"].max()
    cgrid = np.linspace(cmin, cmax, 40)
    r0s = np.linspace(np.quantile(sub["residual"], 0.05), np.quantile(sub["residual"], 0.95), n_r0)
    flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

    def integrate(beta, r0):
        r = float(np.clip(r0, rmin, rmax))
        traj = [r]
        for i in range(len(cgrid) - 1):
            c = cgrid[i]
            dc = float(cgrid[i+1] - cgrid[i])
            x = np.array([1.0, c, r, c**3, r**2, r * c**2], dtype=float)
            g = float(np.clip(x @ beta, -flow_cap, flow_cap))
            r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
            traj.append(r)
        return np.array(traj)

    errs = []
    for r0 in r0s:
        t_ref = integrate(beta_ref, r0)
        t_cmp = integrate(beta_cmp, r0)
        errs.append(math.sqrt(mean_squared_error(t_ref, t_cmp)))
    return float(np.mean(errs))

## Holdout evaluation

In [ ]:
# Hold out one full regime at a time

beta_shared = Y_coef.to_numpy(dtype=float).mean(axis=0)
holdout_rows = []

for i in range(len(coef_df)):
    test_regime = coef_df.iloc[i]["regime_id"]
    train_mask = coef_df["regime_id"] != test_regime
    test_mask = coef_df["regime_id"] == test_regime

    X_train = X_meta.loc[train_mask]
    X_test = X_meta.loc[test_mask]
    Y_train = Y_coef.loc[train_mask]
    Y_test = Y_coef.loc[test_mask]

    candidates = {
        "linear": fit_predict_holdout(X_train, Y_train, X_test, build_linear)[0],
        "ridge": fit_predict_holdout(X_train, Y_train, X_test, build_ridge)[0],
        "mlp_small": fit_predict_holdout(X_train, Y_train, X_test, build_mlp)[0],
    }

    # latent candidates fit only on training regimes
    coef_scaler = StandardScaler()
    Y_train_std = coef_scaler.fit_transform(Y_train.to_numpy())
    pca = PCA(n_components=2, random_state=42)
    Z_train = pca.fit_transform(Y_train_std)

    def recon_latent(model_builder):
        zhat = fit_predict_holdout(X_train, pd.DataFrame(Z_train), X_test, model_builder)[0]
        yhat_std = pca.inverse_transform(np.atleast_2d(zhat))
        yhat = coef_scaler.inverse_transform(yhat_std)
        return yhat[0]

    candidates["latent_linear"] = recon_latent(build_linear)
    candidates["latent_ridge"] = recon_latent(build_ridge)
    candidates["latent_mlp_small"] = recon_latent(build_mlp)

    beta_true = Y_test.to_numpy(dtype=float)[0]
    best_name = min(candidates, key=lambda k: math.sqrt(mean_squared_error(beta_true, candidates[k])))
    beta_meta = candidates[best_name]

    sub = regime_subsets[test_regime]
    y_emp = sub["predicted_flow"].to_numpy(dtype=float)
    pred_shared = predict_with_beta(sub, beta_shared)
    pred_meta = predict_with_beta(sub, beta_meta)
    pred_specific = predict_with_beta(sub, beta_true)

    holdout_rows.append({
        "regime_id": test_regime,
        "best_meta_model": best_name,

        "coef_rmse_shared": math.sqrt(mean_squared_error(beta_true, beta_shared)),
        "coef_rmse_meta": math.sqrt(mean_squared_error(beta_true, beta_meta)),

        "field_rmse_shared": math.sqrt(mean_squared_error(y_emp, pred_shared)),
        "field_rmse_meta": math.sqrt(mean_squared_error(y_emp, pred_meta)),
        "field_rmse_specific": math.sqrt(mean_squared_error(y_emp, pred_specific)),

        "field_corr_shared": safe_corr(y_emp, pred_shared),
        "field_corr_meta": safe_corr(y_emp, pred_meta),
        "field_corr_specific": safe_corr(y_emp, pred_specific),

        "field_ev_shared": explained_variance_score_manual(y_emp, pred_shared),
        "field_ev_meta": explained_variance_score_manual(y_emp, pred_meta),
        "field_ev_specific": explained_variance_score_manual(y_emp, pred_specific),

        "traj_rmse_shared": trajectory_gap(sub, beta_true, beta_shared),
        "traj_rmse_meta": trajectory_gap(sub, beta_true, beta_meta),
    })

holdout_df = pd.DataFrame(holdout_rows)
display(holdout_df.head())

## Holdout summary

In [ ]:
summary = pd.DataFrame([
    {
        "method": "shared_law",
        "coefficient_rmse": holdout_df["coef_rmse_shared"].mean(),
        "field_rmse": holdout_df["field_rmse_shared"].mean(),
        "field_corr": holdout_df["field_corr_shared"].mean(),
        "field_ev": holdout_df["field_ev_shared"].mean(),
        "trajectory_rmse": holdout_df["traj_rmse_shared"].mean(),
    },
    {
        "method": "meta_law_holdout",
        "coefficient_rmse": holdout_df["coef_rmse_meta"].mean(),
        "field_rmse": holdout_df["field_rmse_meta"].mean(),
        "field_corr": holdout_df["field_corr_meta"].mean(),
        "field_ev": holdout_df["field_ev_meta"].mean(),
        "trajectory_rmse": holdout_df["traj_rmse_meta"].mean(),
    },
    {
        "method": "regime_specific",
        "coefficient_rmse": 0.0,
        "field_rmse": holdout_df["field_rmse_specific"].mean(),
        "field_corr": holdout_df["field_corr_specific"].mean(),
        "field_ev": holdout_df["field_ev_specific"].mean(),
        "trajectory_rmse": 0.0,
    },
])

display(summary)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
metrics = ["coefficient_rmse", "field_rmse", "field_corr", "field_ev", "trajectory_rmse"]
titles = ["Coefficient RMSE", "Field RMSE", "Field correlation", "Field explained variance", "Trajectory RMSE"]

for ax, m, t in zip(axes, metrics, titles):
    ax.bar(summary["method"], summary[m])
    ax.set_title(t)
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Best meta-model winner counts

winner_counts = holdout_df["best_meta_model"].value_counts()
display(winner_counts)

plt.figure(figsize=(7,4))
plt.bar(winner_counts.index, winner_counts.values)
plt.title("Best meta-law winner counts across held-out regimes")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Regime-level comparisons

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
pairs = [
    ("coef_rmse_shared", "coef_rmse_meta", "Coefficient RMSE"),
    ("field_rmse_shared", "field_rmse_meta", "Field RMSE"),
    ("traj_rmse_shared", "traj_rmse_meta", "Trajectory RMSE"),
]

for ax, (xs, ys, title) in zip(axes, pairs):
    ax.scatter(holdout_df[xs], holdout_df[ys])
    mn = min(holdout_df[xs].min(), holdout_df[ys].min())
    mx = max(holdout_df[xs].max(), holdout_df[ys].max())
    ax.plot([mn, mx], [mn, mx], linestyle="--")
    ax.set_xlabel("shared-law")
    ax.set_ylabel("meta-law")
    ax.set_title(title)

plt.tight_layout()
plt.show()

In [ ]:
holdout_df["forcing_mode"] = holdout_df["regime_id"].str.split("|").str[2]
traj_force = holdout_df.groupby("forcing_mode")[["traj_rmse_shared", "traj_rmse_meta"]].mean().reset_index()

x = np.arange(len(traj_force))
w = 0.38
plt.figure(figsize=(8,4))
plt.bar(x - w/2, traj_force["traj_rmse_shared"], width=w, label="shared-law")
plt.bar(x + w/2, traj_force["traj_rmse_meta"], width=w, label="meta-law")
plt.xticks(x, traj_force["forcing_mode"])
plt.ylabel("mean holdout trajectory RMSE")
plt.title("Holdout trajectory generalization by forcing mode")
plt.legend()
plt.tight_layout()
plt.show()

## Representative held-out regime

In [ ]:
rep_idx = int(np.argmax(holdout_df["traj_rmse_shared"] - holdout_df["traj_rmse_meta"]))
rep_regime_id = holdout_df.iloc[rep_idx]["regime_id"]

train_mask = coef_df["regime_id"] != rep_regime_id
test_mask = coef_df["regime_id"] == rep_regime_id

X_train = X_meta.loc[train_mask]
X_test = X_meta.loc[test_mask]
Y_train = Y_coef.loc[train_mask]
Y_test = Y_coef.loc[test_mask]
beta_true = Y_test.to_numpy(dtype=float)[0]

candidates = {
    "linear": fit_predict_holdout(X_train, Y_train, X_test, build_linear)[0],
    "ridge": fit_predict_holdout(X_train, Y_train, X_test, build_ridge)[0],
    "mlp_small": fit_predict_holdout(X_train, Y_train, X_test, build_mlp)[0],
}

coef_scaler = StandardScaler()
Y_train_std = coef_scaler.fit_transform(Y_train.to_numpy())
pca = PCA(n_components=2, random_state=42)
Z_train = pca.fit_transform(Y_train_std)

def recon_latent(model_builder):
    zhat = fit_predict_holdout(X_train, pd.DataFrame(Z_train), X_test, model_builder)[0]
    yhat_std = pca.inverse_transform(np.atleast_2d(zhat))
    yhat = coef_scaler.inverse_transform(yhat_std)
    return yhat[0]

candidates["latent_linear"] = recon_latent(build_linear)
candidates["latent_ridge"] = recon_latent(build_ridge)
candidates["latent_mlp_small"] = recon_latent(build_mlp)

best_name = min(candidates, key=lambda k: math.sqrt(mean_squared_error(beta_true, candidates[k])))
beta_meta = candidates[best_name]

sub = regime_subsets[rep_regime_id]
cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
rmin, rmax = sub["residual"].min(), sub["residual"].max()
cgrid = np.linspace(cmin, cmax, 45)
r0s = np.linspace(np.quantile(sub["residual"], 0.1), np.quantile(sub["residual"], 0.9), 8)
flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

def integrate_beta(beta, r0):
    r = float(np.clip(r0, rmin, rmax))
    traj = [r]
    for j in range(len(cgrid) - 1):
        c = cgrid[j]
        dc = float(cgrid[j+1] - cgrid[j])
        x = np.array([1.0, c, r, c**3, r**2, r * c**2], dtype=float)
        g = float(np.clip(x @ beta, -flow_cap, flow_cap))
        r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
        traj.append(r)
    return np.array(traj)

fig, axes = plt.subplots(1, 3, figsize=(15,4), sharey=True)
for r0 in r0s:
    axes[0].plot(cgrid, integrate_beta(beta_shared, r0))
    axes[1].plot(cgrid, integrate_beta(beta_meta, r0))
    axes[2].plot(cgrid, integrate_beta(beta_true, r0))

axes[0].set_title("Shared-law")
axes[1].set_title(f"Holdout meta-law ({best_name})")
axes[2].set_title("Regime-specific")
for ax in axes:
    ax.set_xlabel("condition coordinate c")
axes[0].set_ylabel("residual")
fig.suptitle(f"Representative held-out regime: {rep_regime_id}", y=1.03)
plt.tight_layout()
plt.show()

## Final verdict table

In [ ]:
def verdict(row):
    if row["field_corr_meta"] > row["field_corr_shared"] and row["traj_rmse_meta"] < row["traj_rmse_shared"]:
        if row["coef_rmse_meta"] < row["coef_rmse_shared"]:
            return "meta-law generalizes"
        return "meta-law partly generalizes"
    if row["traj_rmse_meta"] < row["traj_rmse_shared"]:
        return "meta-law helps on trajectories"
    return "shared-law too coarse / hard regime"

holdout_df["verdict"] = holdout_df.apply(verdict, axis=1)
display(holdout_df[[
    "regime_id", "best_meta_model",
    "coef_rmse_shared", "coef_rmse_meta",
    "field_rmse_shared", "field_rmse_meta", "field_rmse_specific",
    "field_corr_shared", "field_corr_meta", "field_corr_specific",
    "traj_rmse_shared", "traj_rmse_meta", "verdict"
]].head(20))

## Working conclusion

Notebook 49 tests whether the metadata-conditioned meta-law from Notebook 48 truly generalizes to unseen regimes.

A strong result is:
- held-out coefficient RMSE improves sharply over a shared-law
- held-out field RMSE and correlation improve over a shared-law
- held-out trajectory RMSE improves substantially over a shared-law
- a simple linear or latent-linear model wins most often

If that pattern holds on your real exports, the next notebook should be:

**Notebook 50 — out-of-family extrapolation and regime interpolation tests**